# 📘 RESOLUCIÓN PASO A PASO Y MANUAL DIDÁCTICO DEL TALLER CAPM
### *Guía Cronológica y Explicativa de Valoración de Activos (DEFI 2) para Científicos de Datos*
**Autor:** Julián Duarte | Universidad Externado de Colombia  
**Caso de Estudio:** Doña Victoria y la búsqueda del 25% de retorno anual sin caídas de capital.

---

## 🎯 Propósito de este Notebook
Si eres estudiante de **Ciencia de Datos**, este notebook está diseñado para explicarte **absolutamente todo desde cero y paso a paso**: cómo se construyó y resolvió este taller, de dónde salieron los datos, qué hace cada empresa analizada, qué es un ETF, qué es una regresión OLS con errores robustos HAC, y por qué llegamos a las conclusiones finales.

Aquí no encontrarás saltos conceptuales. Verás la trayectoria completa dividida en **7 Pasos Cronológicos**.

---
## 📌 PASO 1: DEFINICIÓN DEL ECOSISTEMA DE DATOS Y CONCEPTOS BASE

### 1. ¿De dónde sacamos los datos y cómo los obtuvimos?
Para analizar los activos sin sesgos de muestra, recopilamos un periodo continuo de **499 días de negociación real** (agosto de 2024 a agosto de 2026) a través de tres fuentes principales:
1. **Precios y Volumen de Acciones y ETFs**: Descargados automáticamente mediante la librería `yfinance` en Python conectada a la API de **Yahoo Finance** (usando precios de cierre ajustados por splits y dividendos).
2. **Nasdaq Crypto Index (`NCI`)**: Serie histórica del índice oficial del mercado cripto suministrada por la cátedra en `01_datos/NCI.csv`.
3. **Tasa Libre de Riesgo ($R_f$)**: Datos oficiales descargados de **FRED** (Federal Reserve Bank of St. Louis) de la serie `DGS3MO` (Tasa de rendimiento de los Títulos del Tesoro de EE.UU. a 3 meses), cuyo promedio en el periodo fue del **5.2% anual** (aprox. **0.0142% diario**).

---

### 2. Glosario Didáctico de Términos Clave

#### 🔹 ¿Qué es un ETF (Exchange-Traded Fund / Fondo Cotizado en Bolsa)?
Un **ETF** es un fondo de inversión que cotiza en la bolsa exactamente igual que si fuera una acción individual, pero por dentro contiene una **canasta diversificada de decenas o cientos de empresas de un mismo sector**. En lugar de comprar acciones de 50 empresas tecnológicas una por una, compras 1 título del ETF y obtienes la rentabilidad promedio del sector.

En el taller utilizamos **4 Benchmarks (Mercados de Referencia)**:
* **`XLK` (Technology Select Sector SPDR Fund)**: ETF que agrupa a las mayores empresas de tecnología de EE.UU. (Apple, Microsoft, Nvidia, AMD). Representa el rendimiento del sector tecnológico.
* **`XBI` (SPDR S&P Biotech ETF)**: ETF que agrupa a empresas de biotecnología e investigación farmacéutica. Representa al sector biotecnológico.
* **`XLE` (Energy Select Sector SPDR Fund)**: ETF que reúne a las gigantes petroleras y energéticas (ExxonMobil, Chevron, ConocoPhillips). Representa al sector energético.
* **`NCI` (Nasdaq Crypto Index)**: Índice ponderado que mide la rentabilidad del mercado de criptomonedas (Bitcoin, Ethereum, Solana, etc.).

#### 🔹 ¿Qué es la Tasa Libre de Riesgo ($R_f$)?
Es la rentabilidad garantizada que paga el emisor con cero probabilidad de impago (*default*), en este caso el Gobierno de los Estados Unidos. Es el *baseline* o intercepto mínimo. Si un activo financiero te exige arriesgar tu capital, **debe pagar por encima de la tasa libre de riesgo**; de lo contrario, carece de sentido financiero invertir en él.

---
## 🏢 PASO 2: CARACTERIZACIÓN DETALLADA DE LAS 10 EMPRESAS Y SU ELECCIÓN

El enunciado del taller nos exigió analizar **1 Criptoactivo asignado** (`SOL`) y **9 acciones de bajo precio** (cotizando por debajo de $5 USD, o bajo la excepción permitida de menos de $10 USD) divididas en 3 sectores. 

¿Qué hace exactamente cada una de estas empresas y por qué se escogió?

| Ticker | Empresa | Sector | Cotización USD | ¿Qué hace la empresa? (Modelo de Negocio) | Benchmark de Referencia |
| :--- | :--- | :--- | :---: | :--- | :--- |
| **SOL** | Solana | Criptoactivos | $87.64 | **Criptoactivo de Capa 1.** Red de blockchain de alto rendimiento enfocada en transacciones ultrarrápidas y finanzas descentralizadas (DeFi). | **NCI** (Nasdaq Crypto) |
| **KOPN** | Kopin Corporation | Tecnología | $4.89 | **Óptica Militar y Micropantallas.** Fabrica microdisplays de alta resolución para cascos de pilotos de aviación de combate y dispositivos de realidad aumentada. | **XLK** (Tech ETF) |
| **MVIS** | MicroVision, Inc. | Tecnología | $1.75 | **Sensores LiDAR para Vehículos.** Desarrolla tecnología de escaneo por rayo láser y sensores LiDAR para asistencia de conducción autónoma. | **XLK** (Tech ETF) |
| **LIDR** | AEye, Inc. | Tecnología | $1.26 | **Percepción Espacial LiDAR.** Desarrolla sistemas de visión artificial y software LiDAR adaptativo para transporte e industria. | **XLK** (Tech ETF) |
| **GERN** | Geron Corporation | Biotecnología | $1.55 | **Oncología y Farmacia.** Biofarmacéutica de etapa comercial que desarrolló *Rytelo* para el tratamiento de síndromes mielodisplásicos (cáncer en médula). | **XBI** (Biotech ETF) |
| **CTMX** | CytomX Therapeutics | Biotecnología | $3.29 | **Oncología de Precisión.** Desarrolla terapias con anticuerpos *Probody* diseñados para activarse únicamente dentro de tejidos tumorales. | **XBI** (Biotech ETF) |
| **OCGN** | Ocugen, Inc. | Biotecnología | $1.39 | **Terapias Génicas y Oculares.** Biotecnológica orientada al desarrollo de terapias génicas para ceguera y vacunas. | **XBI** (Biotech ETF) |
| **BORR** | Borr Drilling Limited | Energía | $4.43 | **Perforación Marina Petrolera.** Opera plataformas flotantes de perforación en aguas someras (*jack-up rigs*) para petroleras globales. | **XLE** (Energy ETF) |
| **KLXE** | KLX Energy Services | Energía | $1.63 | **Servicios a Pozos Petroleros.** Proveedor de servicios especializados de intervención, completamiento y herramientas para pozos de hidrocarburos. | **XLE** (Energy ETF) |
| **TTI** | TETRA Technologies | Energía | $7.23 | **Química Petrolera y Agua.** Fluidos de completamiento y tratamiento de aguas para fracturamiento hidráulico (*fracking*). *(Excepción < $10 USD)* | **XLE** (Energy ETF) |

---
### ⚠️ REGLA DE ORO DE LOS BENCHMARKS
* Jamás compares la Beta de Solana con el sector energía, ni la Beta de Kopin con el sector biotecnológico.
* Cada activo fue evaluado **única y exclusivamente contra su ETF sectorial correspondiente**.

---
## 🔄 PASO 3: TRANSFORMACIÓN DE DATOS Y CÁLCULO DE RETORNOS

Los precios de cierre no se pueden usar directamente en modelos de regresión porque tienen **tendencia (no son estacionarios)**. Por ello, transformamos los precios diarios $P_t$ en **Retornos Simples ($R_t$)** o **Logarítmicos ($r_t$)**:

$$R_t = \frac{P_t - P_{t-1}}{P_{t-1}}$$

Veamos el código en Python para cargar los precios limpios y calcular la matriz de retornos:

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['figure.dpi'] = 120

datos_path = '../01_datos/datos_taller_1_julian_duarte.csv'
df_precios = pd.read_csv(datos_path, parse_dates=['Date'], index_col='Date')

print(f"Dataset cargado exitosamente: {df_precios.shape[0]} días de cotización y {df_precios.shape[1]} columnas.")
df_precios[['SOL', 'KOPN', 'XLK', 'GERN', 'XBI', 'BORR', 'XLE', 'Rf_Daily']].head()


---
## 📐 PASO 4: REGRESIÓN CAPM Y EXPLICACIÓN DE OLS ROBUSTO (HAC / WHITE)

### 1. El Modelo CAPM Econométrico
Para cada activo $i$, estimamos la regresión lineal entre el exceso de retorno del activo y el exceso de retorno de su mercado:

$$\Big(R_{i,t} - R_{f,t}\Big) = \alpha_i + \beta_i \cdot \Big(R_{m,t} - R_{f,t}\Big) + \epsilon_{i,t}$$

* **$\beta_i$ (La Beta)**: Mide la pendiente o **sensibilidad del activo ante su sector**.
  * Si $\beta = 1.82$ (caso `KOPN`), la empresa se mueve un 82% más fuerte que el sector tecnológico.
* **$\alpha_i$ (El Alfa de Jensen)**: Mide el intercepto o la **generación/destrucción neta de valor** tras controlar por riesgo.
  * Si $\alpha > 0$, el activo superó la exigencia del mercado.
  * Si $\alpha < 0$, el activo fue una trampa de riesgo.

---

### 2. Explicación Didáctica: ¿Qué es OLS y por qué usamos Errores Robusto HAC/White?

#### 🔸 Mínimos Cuadrados Ordinarios (OLS - Ordinary Least Squares)
Es el algoritmo básico de regresión lineal. Busca los valores de $\alpha$ y $\beta$ que minimizan la suma de errores al cuadrado. Sin embargo, OLS asume dos supuestos ideales que **NUNCA se cumplen en finanzas**:
1. **Homocedasticidad**: Asume que la variabilidad de los errores es constante en el tiempo.
2. **No Autocorrelación**: Asume que el error de hoy no depende del error de ayer.

#### 🔸 ¿Qué ocurre en la realidad financiera? (Heterocedasticidad y Clusters de Volatilidad)
En los mercados hay días de pánico con caídas del 15% y semanas tranquilas con variaciones del 0.2%. Esto se llama **Heterocedasticidad**. Si usas OLS estándar, el algoritmo calcula mal los errores estándar, lo que resulta en **p-valores falsos** que te hacen creer que una variable es significativamente rentable cuando en realidad no lo es.

#### 🔸 La Solución: Errores Robusto HAC (Newey-West) y White (HC1)
Para resolver esto, aplicamos correcciones de la matriz de covarianza:
* **HAC (Heteroskedasticity and Autocorrelation Consistent / Newey-West)**: Ajusta la matriz de covarianza considerando tanto la volatilidad cambiante como el efecto de arrastre de días anteriores (lags).
* **White (HC1)**: Ajusta los errores estándar protegiéndonos contra la heterocedasticidad extrema de las penny stocks.

Veamos una demostración en Python ejecutando la regresión OLS Robusta HAC para Kopin Corp (`KOPN`):

In [ ]:
import statsmodels.api as sm

exceso_kopn = df_precios['KOPN_Ret'] - df_precios['Rf_Daily']
exceso_xlk = df_precios['XLK_Ret'] - df_precios['Rf_Daily']

X = sm.add_constant(exceso_xlk)
modelo_hac = sm.OLS(exceso_kopn, X).fit(cov_type='HAC', cov_kwds={'maxlags': 5})

alpha_anual = modelo_hac.params[0] * 252
beta_estimada = modelo_hac.params[1]
r_cuadrado = modelo_hac.rsquared

print("=== REGRESIÓN CAPM ROBUSTA HAC (KOPN vs XLK) ===")
print(f"Alfa Anualizado (Jensen): {alpha_anual*100:.2f}% (p-valor: {modelo_hac.pvalues[0]:.4f})")
print(f"Beta (Sensibilidad):       {beta_estimada:.4f} (p-valor: {modelo_hac.pvalues[1]:.4e})")
print(f"R-cuadrado (Explicado):    {r_cuadrado*100:.2f}%")


---
## 🧪 PASO 5: AJUSTES DE ILIQUIDEZ (DIMSON BETA) Y WINSORIZACIÓN

### 1. ¿Qué es la Beta de Dimson?
En acciones pequeñas o de poco volumen (*penny stocks*), puede ocurrir que una noticia del sector tecnológico no mueva el precio de la acción inmediatamente a las 10:00 AM, sino que el impacto se refleje **al día siguiente** por falta de compradores/vendedores continuos.

La **Beta de Dimson** corrige esto agregando retornos del mercado rezagados ($R_{m, t-1}$):

$$R_{i,t} - R_{f,t} = \alpha + \beta_0 (R_{m,t} - R_{f,t}) + \beta_1 (R_{m,t-1} - R_{f,t-1}) + \epsilon_t$$
$$\beta_{\text{Dimson Sum}} = \beta_0 + \beta_1$$

---

### 2. ¿Qué es la Winsorización de Datos?
Consiste en acotar los **outliers o valores extremos** (por ejemplo, el 1% de las caídas o subidas más brutales causadas por rumores) para evitar que una sola vela distorsione la pendiente de la regresión OLS.

---
## 📊 PASO 6: CÁLCULO DE MÉTRICAS DE RIESGO DE COLA E ILIQUIDEZ

Para evaluar el sufrimiento real de Doña Victoria en el proceso, calculamos 7 métricas de riesgo:

1. **Volatilidad Anual ($\sigma_{\text{anual}}$)**: $\sigma_{\text{diaria}} \times \sqrt{K}$. Mide la dispersión total del activo.
2. **Sharpe Ratio**: $\frac{R_i - R_f}{\sigma_i}$. Premio por unidad de riesgo total.
3. **Treynor Ratio**: $\frac{R_i - R_f}{\beta_i}$. Premio por unidad de riesgo sistemático.
4. **Sortino Ratio**: $\frac{R_i - R_f}{\sigma_{\text{downside}}}$. Premio considerando únicamente las caídas (volatilidad mala).
5. **Max Drawdown (MDD)**: La peor caída acumulada desde el punto más alto hasta el más bajo.
6. **VaR 95% y CVaR 95% (Conditional VaR / Expected Shortfall)**: El CVaR mide el **promedio de pérdidas en el 5% de los peores días** (los días negros del mercado).
7. **Ratio de Iliquidez de Amihud**: Mide el impacto de precio por cada dólar negociado: $\text{Amihud} = \frac{|R_t|}{\text{Volumen Monetario}_t}$. Si es alto, es una trampa de salida.

Carguemos la **Tabla Maestra Consolidada** de resultados generada en el taller:

In [ ]:
master_path = '../01_datos/tabla_maestra_resultados.csv'
df_master = pd.read_csv(master_path)

cols_show = ['Ticker', 'Family', 'Benchmark', 'Beta', 'Alpha_Annual', 'Observed_Return', 'CAPM_Expected_Return', 'MDD', 'CVaR_95_Daily', 'Pct_Zero_Returns']
df_master[cols_show]


### Visualización Comparativa: Alfa de Jensen vs Beta de Mercado

In [ ]:
plt.figure(figsize=(10, 6))
colors = {'Cripto': '#8e44ad', 'Tecnologia': '#27ae60', 'Biotecnologia': '#c0392b', 'Energia': '#d35400'}

for sector, group in df_master.groupby('Family'):
    plt.scatter(group['Beta'], group['Alpha_Annual'] * 100, label=sector, s=140, color=colors[sector], alpha=0.85, edgecolors='black')

for _, row in df_master.iterrows():
    plt.annotate(row['Ticker'], (row['Beta'] + 0.02, row['Alpha_Annual'] * 100 + 1.5), fontsize=11, fontweight='bold')

plt.axhline(0, color='gray', linestyle='--', label='Línea de Alfa Cero (Cumplimiento CAPM)')
plt.title('Recompensa Neta (Alfa de Jensen) vs Sensibilidad (Beta)', fontsize=14, fontweight='bold')
plt.xlabel('Beta de Mercado (Sensibilidad Sectorial)', fontsize=12)
plt.ylabel('Alfa Anualizado (%)', fontsize=12)
plt.legend(title='Sector', loc='upper left')
plt.tight_layout()
plt.show()


: 

---
## 👑 PASO 7: EVALUACIÓN DE RESULTADOS Y VEREDICTO FINAL PARA DOÑA VICTORIA

### 1. El Resumen del Veredicto Empírico
* **Solo 4 de los 10 activos pagaron Alfa positivo real**: `KOPN` (+80.7%), `CTMX` (+72.8%), `LIDR` (+60.1%) y `TTI` (+29.8%).
* **6 activos destruyeron patrimonio respecto a lo exigido**: `MVIS` ($\alpha = -82.9\%$, Caída máxima del **-93.9%**), `KLXE` ($\alpha = -67.8\%$), `GERN` ($\alpha = -64.1\%$), `BORR` ($\alpha = -29.5\%$) y `SOL` ($\alpha = -28.9\%$).

---

### 2. Respuesta a Doña Victoria: ¿Es posible ganar 25% sin caídas de capital?
**LA RESPUESTA ES NO.**
En los mercados financieros existe un **Trade-off Infranqueable entre Riesgo y Retorno**. Para acceder a rentabilidades del 50% al 120% en los activos ganadores, el inversionista **estuvo expuesto a caídas temporales de entre el -57% y el -83%**.

---

### 🛡️ TRES MANDATOS DE INVERSIÓN SUGERIDOS

| Mandato | Perfil | Composición Sugerida | Retorno Esperado | Caída Máxima Est. |
| :--- | :--- | :--- | :--- | :--- |
| **1. Conservador (Doña Victoria)** | Preservación Familiar | 70% Treasuries DGS3MO (5.2%) + 30% ETFs Sectoriales (`XLE`/`XLK`) | **7.5% - 9.5% Anual** | **< -8%** |
| **2. Moderado (Equilibrado)** | Crecimiento con Control | 50% ETFs Sectoriales + 30% Treasuries + 20% Selección Alfa (`TTI`, `KOPN`) | **14% - 18% Anual** | **< -22%** |
| **3. Agresivo (Especulativo)** | Búsqueda de Alfa Cuantitativo | 50% Ganadores de Alfa (`KOPN`, `CTMX`, `LIDR`) + 30% ETFs + 20% `SOL` | **> 30% Anual** | **> -60%** |
